# 02 — Vision Model Training
Fine-tune ImageNet-pretrained convolutional backbones for crop disease diagnosis across 14 shared classes.

**Ablations & Experiments**:
1. **Sanity Check**: 1-epoch real-mode test (`--epochs 1 --max_images_per_class 50`)
2. **Primary Model**: EfficientNet-B0 on PlantVillage (with data augmentation + cosine schedule + AMP)
3. **Ablation 1 (Backbones)**: ResNet-18 vs MobileNetV3 vs EfficientNet-B0
4. **Ablation 2 (Augmentation)**: EfficientNet-B0 with vs without data augmentation
5. **Ablation 3 (Domain Gap)**: PlantVillage only vs PlantVillage + PlantDoc-train tested on PlantDoc OOD
6. **Multi-Seed Stability**: Running with `--seed 42`, `--seed 43`, `--seed 44`

**Prerequisites**: Run `01_data.ipynb` first.

## 0. Environment Setup & Data Copy to Local Runtime
Mount Google Drive, install dependencies, and copy dataset splits to fast local disk (`/content/project/data`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, torch
DRIVE_ROOT = '/content/drive/MyDrive/farmer_advisory_chatbot'
os.makedirs(f"{DRIVE_ROOT}/checkpoints/vision", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs/vision", exist_ok=True)

# Set working directory
%cd /content/project
!pip install -r requirements.txt -q

# Check CUDA device
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. Switch Colab runtime to GPU (T4) before training.")

# Copy data from Drive to fast local /content/project/data directory
if os.path.exists(f"{DRIVE_ROOT}/data"):
    shutil.copytree(f"{DRIVE_ROOT}/data", "data", dirs_exist_ok=True)
    print("Successfully copied dataset splits from Drive to local /content/project/data")

# Audit: ensure real dataset CSVs are present and not smoke data
with open("data/vision_train.csv", "r") as f:
    first_line = f.readline()
    second_line = f.readline()
    if "smoke_img" in second_line:
        print("WARNING: Detected synthetic smoke data in data/vision_train.csv!")
        print("Regenerating real splits now using download_plantvillage.py --real...")
        !python data_prep/download_plantvillage.py --real --output_dir data/plantvillage
        !python data_prep/download_plantdoc.py --real --output_dir data/plantdoc
        !python data_prep/prepare_splits.py --pv_dir data/plantvillage --pd_dir data/plantdoc
    else:
        print("Verified: real dataset splits ready for training.")

## 1. Fast Real-Mode Sanity Test
Runs 1 epoch with a 50-image-per-class cap to verify the entire CUDA pipeline, AMP, logging, and Drive persistence within ~2 minutes.

In [ ]:
!python train_vision.py --config configs/vision_efficientnet_b0.yaml \
    --epochs 1 \
    --max_images_per_class 50 \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 2. Primary Model: EfficientNet-B0 (Full Run)
Full 20-epoch training on PlantVillage with data augmentation, cosine LR decay, and weighted loss.

In [ ]:
!python train_vision.py --config configs/vision_efficientnet_b0.yaml \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 3. Backbone Ablation 1: ResNet-18

In [ ]:
!python train_vision.py --config configs/vision_resnet18.yaml \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 4. Backbone Ablation 2: MobileNetV3-Small

In [ ]:
!python train_vision.py --config configs/vision_mobilenetv3.yaml \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 5. Augmentation Ablation: Without Augmentation

In [ ]:
!python train_vision.py --config configs/vision_no_aug.yaml \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 6. Domain Gap Ablation: PV + PlantDoc Train

In [ ]:
!python train_vision.py --config configs/vision_domain_gap.yaml \
    --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 7. Multi-Seed Stability (Seeds 43 and 44)

In [ ]:
# Run additional seeds for primary model to compute mean and standard error
!python train_vision.py --config configs/vision_efficientnet_b0.yaml --seed 43 --output_root /content/drive/MyDrive/farmer_advisory_chatbot
!python train_vision.py --config configs/vision_efficientnet_b0.yaml --seed 44 --output_root /content/drive/MyDrive/farmer_advisory_chatbot

## 8. Summary Comparison Table (Parameters, Latency, Accuracies)

In [ ]:
import glob, json, pandas as pd

log_pattern = f"{DRIVE_ROOT}/logs/vision/*_profile.json"
profiles = []
for pf in glob.glob(log_pattern):
    with open(pf, "r") as f:
        d = json.load(f)
        if d.get("is_smoke", False):
            continue  # Exclude smoke test files
        profiles.append({
            "Run": d["run_name"],
            "Backbone": d.get("backbone_variant", d["backbone"]),
            "Seed": d.get("seed", 42),
            "Parameters": f"{d['profile']['total_parameters']:,}",
            "CPU Latency (ms)": d["profile"]["mean_latency_ms"],
            "Best Val Loss": round(d["best_val_loss"], 4)
        })

summary_df = pd.DataFrame(profiles)
display(summary_df)
summary_df.to_csv(f"{DRIVE_ROOT}/logs/vision/ablation_vision_summary.csv", index=False)